### Conceptual Answers


**1. Difference between "Love" and "love":**
- Case-sensitive systems treat them as different tokens. Without normalization,
this leads to feature fragmentation. Lowercasing ensures consistency.

**2. If stopwords are not removed:**
- Increased noise
- Larger vocabulary size
- Reduced model efficiency
- Possible performance degradation

**3. When removing stopwords is harmful:**
- Sentiment analysis: "not good" -> removing "not" changes meaning
- Question answering: "what is AI" -> removing "what" affects intent

**4. Stemming vs Lemmatization:**
- Stemming: rule-based truncation (running -> run)
- Lemmatization: dictionary-based normalization (better linguistic accuracy)


### Advanced Preprocessing Function

In [1]:
import re
import string

def normalize_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

def remove_urls_emails(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    return text

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_non_ascii(text):
    return re.sub(r'[^\x00-\x7F]+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def filter_tokens(tokens):
    return [t for t in tokens if len(t) > 2 or t in ["no", "not"]]

def preprocess_text(text):
    # Error handling
    if not isinstance(text, str) or text.strip() == "":
        return [], ""

    text = remove_urls_emails(text)
    text = remove_numbers(text)
    text = text.lower()
    text = remove_non_ascii(text)
    text = normalize_repeated_chars(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)

    tokens = text.split()
    tokens = filter_tokens(tokens)

    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

### Stress Testing

In [2]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for sentence in test_sentences:
    tokens, clean = preprocess_text(sentence)

    results.append({
        "original": sentence,
        "tokens": tokens,
        "cleaned": clean
    })

for r in results:
    print("Original:", r["original"])
    print("Tokens:", r["tokens"])
    print("Cleaned Sentence:", r["cleaned"])
    print("-" * 60)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
------------------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
------------------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
------------------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call']
Cleaned Sentence: call
------------------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
------------------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned Sentence: visit now
-------------------------------------

### Token Analytics

In [3]:
def compute_analytics(tokens):
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = (
        sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0
    )
    return total_tokens, unique_tokens, avg_token_length


analytics_results = []

for r in results:
    tokens = r["tokens"]
    total, unique, avg_len = compute_analytics(tokens)

    analytics_results.append({
        "total_tokens": total,
        "unique_tokens": unique,
        "avg_length": round(avg_len, 2)
    })

for i, a in enumerate(analytics_results):
    print(f"Sentence {i+1}")
    print(a)
    print("-" * 40)


# Analysis (deterministic explanation)
print("Analysis:")
print("Most noisy sentence: sentences containing numbers, emojis, repeated characters.")
print("Most meaningful tokens: sentences with clear semantic words after preprocessing.")

Sentence 1
{'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.0}
----------------------------------------
Sentence 2
{'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 6.5}
----------------------------------------
Sentence 3
{'total_tokens': 3, 'unique_tokens': 3, 'avg_length': 5.33}
----------------------------------------
Sentence 4
{'total_tokens': 1, 'unique_tokens': 1, 'avg_length': 4.0}
----------------------------------------
Sentence 5
{'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.25}
----------------------------------------
Sentence 6
{'total_tokens': 2, 'unique_tokens': 2, 'avg_length': 4.0}
----------------------------------------
Sentence 7
{'total_tokens': 3, 'unique_tokens': 3, 'avg_length': 3.0}
----------------------------------------
Sentence 8
{'total_tokens': 1, 'unique_tokens': 1, 'avg_length': 3.0}
----------------------------------------
Sentence 9
{'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.5}
----------------------------------------

### Frequency Analysis

In [4]:
from collections import Counter

all_tokens = []
for r in results:
    all_tokens.extend(r["tokens"])

token_counter = Counter(all_tokens)

top_10 = token_counter.most_common(10)
least_5 = token_counter.most_common()[-5:]

print("Top 10 Frequent Words:")
print(top_10)

print("\nTop 5 Least Frequent Words:")
print(least_5)

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


### Full Pipeline

In [5]:
def full_pipeline(text_list):
    if not isinstance(text_list, list):
        raise ValueError("Input must be a list of strings")

    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


pipeline_output = full_pipeline(test_sentences)

print("Pipeline Output:")
print(pipeline_output)

Pipeline Output:
{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this']}


### Error Handling

In [6]:
edge_cases = [
    "",
    "😂😂😂",
    "123456789",
    None
]

for case in edge_cases:
    tokens, clean = preprocess_text(case)
    print("Input:", case)
    print("Tokens:", tokens)
    print("Cleaned:", clean)
    print("-" * 40)

Input: 
Tokens: []
Cleaned: 
----------------------------------------
Input: 😂😂😂
Tokens: []
Cleaned: 
----------------------------------------
Input: 123456789
Tokens: []
Cleaned: 
----------------------------------------
Input: None
Tokens: []
Cleaned: 
----------------------------------------
